# V2 Phase 7 — Colab GPU smoke (Qwen3-8B)

**Before running:** Runtime → Change runtime type → **GPU**.

## 1. Mount Google Drive and find V2

Put `V2/` on Google Drive, run the cell below, and approve the mount prompt.
The cell searches My Drive for `config/experiment.yaml` + `scripts/smoke_generate.py`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')


def is_v2(path: Path) -> bool:
    return (path / 'config' / 'experiment.yaml').is_file() and (
        path / 'scripts' / 'smoke_generate.py'
    ).is_file()


def find_v2_on_drive(max_depth: int = 8):
    roots = [Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives')]
    found = []
    for root in roots:
        if not root.is_dir():
            continue
        stack = [(root, 0)]
        while stack:
            current, depth = stack.pop()
            if is_v2(current):
                found.append(current.resolve())
            if depth >= max_depth:
                continue
            try:
                for child in current.iterdir():
                    if child.is_dir() and not child.name.startswith('.'):
                        stack.append((child, depth + 1))
            except OSError:
                pass
    if not found:
        return None
    found.sort(key=lambda p: (p.name != 'V2', len(p.parts), str(p)))
    return found[0]


V2_ROOT = find_v2_on_drive()
if V2_ROOT is None:
    raise FileNotFoundError('V2 not found on Google Drive. Copy V2/ to My Drive and re-run.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('V2_ROOT:', V2_ROOT)

## 2. Install dependencies (Colab GPU)

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Smoke test (llama_cpp)

In [ ]:
!PYTHONPATH=. python scripts/smoke_generate.py --backend llama_cpp --notebook notebooks/colab_phase7_smoke.ipynb

In [ ]:
# Fallback if llama_cpp fails:
# !PYTHONPATH=. python scripts/smoke_generate.py --backend transformers --notebook notebooks/colab_phase7_smoke.ipynb

## 4. Confirm artefacts

- `results/config/phase7_runtime_fingerprint.json`
- `results/config/phase7_smoke_test.json`

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase7_runtime_fingerprint.json')
smoke = Path('results/config/phase7_smoke_test.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', repr(data.get('actual')))